# Week 7 Exercise: Confidence-Aware QLoRA Pricer

This notebook fine-tunes an open-source model for product price prediction using QLoRA, then compares two inference modes:

- greedy price decoding
- confidence-aware weighted top-k decoding

It finishes with MAE/RMSE evaluation and category-level error analysis.


In [2]:
# If needed, install dependencies in Colab or a fresh venv
!pip -q install datasets transformers peft trl bitsandbytes accelerate sentencepiece huggingface_hub scikit-learn


pyenv: version `3.12.12' is not installed (set by /Users/andela/projects/llm_engineering/.python-version)
pyenv: pip: command not found

The `pip' command exists in these Python versions:
  2.7.18

Note: See 'pyenv help global' for tips on allowing both
      python2 and python3 to be found.


In [12]:
# Imports
import math
import os
import re
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import mean_absolute_error, mean_squared_error
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [13]:
# Config
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_NAME = "ed-donner/items_prompts_lite"
OUTPUT_DIR = "week7_qwen_confidence_pricer"

MAX_SEQ_LENGTH = 512
TRAIN_LIMIT = 5000
EVAL_LIMIT = 200

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

NUM_EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4

HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))


GPU available: False


In [14]:
# Load prompt dataset
dataset = load_dataset(DATASET_NAME)
train_dataset = dataset["train"]
eval_dataset = dataset["val"] if "val" in dataset else dataset["validation"]
test_dataset = dataset["test"]

if TRAIN_LIMIT:
    train_dataset = train_dataset.select(range(min(TRAIN_LIMIT, len(train_dataset))))
    eval_dataset = eval_dataset.select(range(min(max(100, TRAIN_LIMIT // 10), len(eval_dataset))))

def to_sft_text(example):
    return {"text": example["prompt"] + example["completion"]}

train_sft = train_dataset.map(to_sft_text)
eval_sft = eval_dataset.map(to_sft_text)

print("Train rows:", len(train_sft))
print("Eval rows:", len(eval_sft))
print("Test rows:", len(test_dataset))
print()
print("Sample prompt:")
print(test_dataset[0]["prompt"][:500])
print()
print("Sample completion:", test_dataset[0]["completion"])


Train rows: 5000
Eval rows: 500
Test rows: 1000

Sample prompt:
What does this cost to the nearest dollar?

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

Pr

Sample completion: 219.0


In [15]:
# Tokenizer + 4-bit model setup
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = None
model_kwargs = {"trust_remote_code": True}
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model_kwargs["quantization_config"] = bnb_config
    model_kwargs["device_map"] = "auto"

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
base_model.config.use_cache = False

if torch.cuda.is_available():
    base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [16]:
# Trainer setup
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=False,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_sft,
    eval_dataset=eval_sft,
    processing_class=tokenizer,
)

trainer


Adding EOS to train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
# Training
# Uncomment when you are ready to fine-tune on a GPU runtime.
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
/Users/andela/projects/llm_engineering/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss


In [ ]:
# Optional: reload saved adapter in a fresh session
# base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **model_kwargs)
# fine_tuned_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
# fine_tuned_model.eval()
# model = fine_tuned_model


In [ ]:
# Confidence-aware inference helpers
def model_device(model_to_use):
    return next(model_to_use.parameters()).device

def extract_price(text):
    match = re.search(r"[-+]?\d+(?:\.\d+)?", text.replace(",", ""))
    return float(match.group()) if match else None

def extract_category(prompt):
    match = re.search(r"Category:\s*(.+)", prompt)
    return match.group(1).strip() if match else "Unknown"

@torch.inference_mode()
def greedy_price_predict(prompt, model_to_use=None):
    model_to_use = model_to_use or trainer.model
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device(model_to_use))
    output = model_to_use.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return extract_price(completion) or 0.0, completion

@torch.inference_mode()
def weighted_topk_price_predict(prompt, k=5, model_to_use=None):
    model_to_use = model_to_use or trainer.model
    inputs = tokenizer(prompt, return_tensors="pt").to(model_device(model_to_use))
    output = model_to_use.generate(
        **inputs,
        max_new_tokens=8,
        num_beams=k,
        num_return_sequences=k,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    prompt_len = inputs["input_ids"].shape[1]
    completions = tokenizer.batch_decode(output.sequences[:, prompt_len:], skip_special_tokens=True)
    probs = F.softmax(output.sequences_scores.float().cpu(), dim=0).tolist()

    candidates = []
    for completion, prob in zip(completions, probs):
        price = extract_price(completion)
        if price is not None and price > 0:
            candidates.append({"price": price, "prob": prob, "completion": completion.strip()})

    if not candidates:
        return 0.0, []

    total_prob = sum(c["prob"] for c in candidates)
    weighted_price = sum(c["price"] * (c["prob"] / total_prob) for c in candidates)
    return weighted_price, candidates


In [ ]:
# Evaluation helpers
def evaluate_predictor(name, predictor, dataset_split, size=EVAL_LIMIT):
    subset = dataset_split.select(range(min(size, len(dataset_split))))
    actuals = []
    preds = []
    rows = []

    for row in subset:
        prediction, detail = predictor(row["prompt"])
        actual = extract_price(row["completion"]) or 0.0
        actuals.append(actual)
        preds.append(float(prediction))
        rows.append(
            {
                "category": extract_category(row["prompt"]),
                "actual": actual,
                "pred": float(prediction),
                "abs_error": abs(float(prediction) - actual),
                "detail": detail,
                "prompt": row["prompt"],
            }
        )

    mae = mean_absolute_error(actuals, preds)
    rmse = math.sqrt(mean_squared_error(actuals, preds))
    return {"name": name, "mae": mae, "rmse": rmse, "rows": rows}

def print_category_report(rows, top_n=10):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["category"]].append(row["abs_error"])

    summary = []
    for category, errors in grouped.items():
        summary.append((category, sum(errors) / len(errors), len(errors)))
    summary.sort(key=lambda item: (-item[1], -item[2], item[0]))

    print("Category error report")
    for category, avg_error, count in summary[:top_n]:
        print(f"{category:30} avg_abs_error=${avg_error:8.2f}  n={count}")

def print_worst_rows(rows, top_n=5):
    rows = sorted(rows, key=lambda row: row["abs_error"], reverse=True)
    for row in rows[:top_n]:
        print("\n---")
        print("Category:", row["category"])
        print("Actual:", round(row["actual"], 2), "Pred:", round(row["pred"], 2), "AbsErr:", round(row["abs_error"], 2))
        print("Prompt snippet:")
        print(row["prompt"][:260] + ("..." if len(row["prompt"]) > 260 else ""))
        print("Detail:", row["detail"])


In [ ]:
# Run evaluation after training
# greedy_result = evaluate_predictor("greedy", greedy_price_predict, test_dataset)
# weighted_result = evaluate_predictor("weighted_topk", weighted_topk_price_predict, test_dataset)
# print(greedy_result["name"], "MAE=", round(greedy_result["mae"], 2), "RMSE=", round(greedy_result["rmse"], 2))
# print(weighted_result["name"], "MAE=", round(weighted_result["mae"], 2), "RMSE=", round(weighted_result["rmse"], 2))
# print_category_report(weighted_result["rows"])
# print_worst_rows(weighted_result["rows"])
